In [15]:
import pandas as pd
import numpy as np

# Load the datasets
weather_df = pd.read_csv('SriLanka_Weather_Dataset.csv')
location_df = pd.read_csv('sri_lanka_heritage_1000 (3).csv')

print("\nWEATHER DATASET COLUMNS:")
for i, col in enumerate(weather_df.columns, 1):
    print(f"{i}. {col}")

print("\nLOCATION DATASET COLUMNS:")
for i, col in enumerate(location_df.columns, 1):
    print(f"{i}. {col}")


# Show dataset shapes
print("\nDATASET INFORMATION:")
print(f"Weather dataset shape: {weather_df.shape} (rows, columns)")
print(f"Location dataset shape: {location_df.shape} (rows, columns)")


WEATHER DATASET COLUMNS:
1. time
2. weathercode
3. temperature_2m_max
4. temperature_2m_min
5. temperature_2m_mean
6. apparent_temperature_max
7. apparent_temperature_min
8. apparent_temperature_mean
9. sunrise
10. sunset
11. shortwave_radiation_sum
12. precipitation_sum
13. rain_sum
14. snowfall_sum
15. precipitation_hours
16. windspeed_10m_max
17. windgusts_10m_max
18. winddirection_10m_dominant
19. et0_fao_evapotranspiration
20. latitude
21. longitude
22. elevation
23. country
24. city

LOCATION DATASET COLUMNS:
1. name
2. address
3. lat
4. lng
5. rating
6. types
7. reviews
8. photos
9. description
10. district
11. keyword

DATASET INFORMATION:
Weather dataset shape: (147480, 24) (rows, columns)
Location dataset shape: (941, 11) (rows, columns)


In [12]:
import pandas as pd
import numpy as np

weather = pd.read_csv('SriLanka_Weather_Dataset.csv')
location= pd.read_csv('sri_lanka_heritage_1000 (3).csv')


weather_drop_cols = [
    'country',
    'et0_fao_evapotranspiration',
    'snowfall_sum',
    'shortwave_radiation_sum'
]

weather = weather.drop(columns=weather_drop_cols)

location_drop_cols = [
    'rating',
    'types',
    'reviews',
    'photos'
]

location = location.drop(columns=location_drop_cols)

weather['time'] = pd.to_datetime(weather['time'])

weather = weather.rename(columns={
    'latitude': 'station_lat',
    'longitude': 'station_lng'
})

location = location.rename(columns={
    'lat': 'location_lat',
    'lng': 'location_lng'
})

# Fill numeric missing values with column mean
num_cols_weather = weather.select_dtypes(include='number').columns
weather[num_cols_weather] = weather[num_cols_weather].fillna(
    weather[num_cols_weather].mean()
)

import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + \
        np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2

    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


stations = weather[['station_lat', 'station_lng', 'city', 'elevation']].drop_duplicates().reset_index(drop=True)

nearest_station_list = []

for _, loc in location.iterrows():
    distances = haversine(
        loc['location_lat'],
        loc['location_lng'],
        stations['station_lat'],
        stations['station_lng']
    )

    nearest_idx = distances.idxmin()
    nearest_station = stations.loc[nearest_idx]

    nearest_station_list.append({
        'name': loc['name'],
        'address': loc['address'],
        'district': loc['district'],
        'keyword': loc['keyword'],
        'location_lat': loc['location_lat'],
        'location_lng': loc['location_lng'],
        'station_lat': nearest_station['station_lat'],
        'station_lng': nearest_station['station_lng'],
        'city': nearest_station['city'],
        'elevation': nearest_station['elevation'],
        'distance_km': distances.min()
    })

location_station_map = pd.DataFrame(nearest_station_list)

final_df = pd.merge(
    weather,
    location_station_map,
    on=['station_lat', 'station_lng'],
    how='inner'
)

final_df.to_csv('final_weather_location_dataset.csv', index=False)

print(final_df.shape)
final_df.head()





(5786132, 29)


,time,weathercode,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,sunrise,sunset,...,city_x,name,address,district,keyword,location_lat,location_lng,city_y,elevation_y,distance_km
0,2010-01-01,2,30.0,22.7,26.1,34.4,25.2,29.2,2010-01-01T00:52,2010-01-01T12:35,...,Colombo,Maha Kali Amman Temple,"Kimbulawala, XV88XQ6, Colombo, Sri Lanka",Colombo,ancient temple,6.967414,79.866898,Colombo,16.0,5.145086
1,2010-01-01,2,30.0,22.7,26.1,34.4,25.2,29.2,2010-01-01T00:52,2010-01-01T12:35,...,Colombo,Hindu Temple Murugan Kovil,"XVMH5CM, Wattala, Sri Lanka",Colombo,hindu kovil,6.982959,79.878540,Colombo,16.0,3.032693
2,2010-01-01,2,30.0,22.7,26.1,34.4,25.2,29.2,2010-01-01T00:52,2010-01-01T12:35,...,Colombo,Iyngaran Pillayar Kovil,"01300 Kotahena St, Colombo 01300, Sri Lanka",Colombo,hindu kovil,6.950052,79.862770,Colombo,16.0,6.908412
3,2010-01-01,2,30.0,22.7,26.1,34.4,25.2,29.2,2010-01-01T00:52,2010-01-01T12:35,...,Colombo,Sri Ananda Aiyappan Temple,"XV58QC2, Colombo, Sri Lanka",Colombo,hindu kovil,6.959395,79.866013,Colombo,16.0,5.869624
4,2010-01-01,2,30.0,22.7,26.1,34.4,25.2,29.2,2010-01-01T00:52,2010-01-01T12:35,...,Colombo,Mukadam Shivan Temple,"XV88VMH, Temple Rd, Colombo, Sri Lanka",Colombo,hindu kovil,6.967182,79.866729,Colombo,16.0,5.176515


In [1]:
import pandas as pd

# Load final dataset
df = pd.read_csv('final_weather_location_dataset.csv')

# Columns to KEEP (method + tourism focused)
columns_to_keep = [
    # Methodology (nearest station linking)
    'station_lat',
    'station_lng',
    'distance_km',

    # Time
    'time',

    # Temperature (tourist-relevant)
    'temperature_2m_max',
    'temperature_2m_min',
    'temperature_2m_mean',

    # Weather comfort
    'apparent_temperature_mean',
    'weathercode',

    # Rain & precipitation
    'precipitation_sum',
    'rain_sum',
    'precipitation_hours',

    # Wind
    'windspeed_10m_max',

    # Day planning
    'sunrise',
    'sunset',

    # Location info
    'name',
    'address',
    'district',
    'keyword',
    'location_lat',
    'location_lng',

    # Reference climate info
    'city_x',
    'elevation_x'
]

# Keep only these columns
df_clean = df[columns_to_keep]

print("Final columns:")
print(df_clean.columns.tolist())
print("Final shape:", df_clean.shape)

# Save cleaned dataset
df_clean.to_csv('final_weather_location_tourism_ready.csv', index=False)


Final columns:
['station_lat', 'station_lng', 'distance_km', 'time', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_mean', 'weathercode', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'windspeed_10m_max', 'sunrise', 'sunset', 'name', 'address', 'district', 'keyword', 'location_lat', 'location_lng', 'city_x', 'elevation_x']
Final shape: (5786132, 23)


In [12]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import requests
from geopy.geocoders import Nominatim

df = pd.read_csv('final_weather_location_tourism_ready.csv')

def weather_category(code):
    if code in [0, 1]:
        return 'Clear'
    elif code in [2, 3]:
        return 'Cloudy'
    elif code in [45, 48]:
        return 'Fog'
    elif code in [51, 53, 55, 61, 63, 65]:
        return 'Rain'
    elif code in [80, 81, 82]:
        return 'Heavy Rain'
    else:
        return 'Other'


df['weather_condition'] = df['weathercode'].apply(weather_category)
df['time'] = pd.to_datetime(df['time'])
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day
df['dayofweek'] = df['time'].dt.dayofweek

feature_cols = [
    'location_lat', 'location_lng', 'elevation_x',
    'station_lat', 'station_lng', 'distance_km',
    'month', 'day', 'dayofweek',
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'apparent_temperature_mean',
    'precipitation_sum', 'rain_sum', 'precipitation_hours',
    'windspeed_10m_max'
]

X = df[feature_cols]
y = df['weather_condition']

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    class_weight="balanced",
    verbose=-1
)

model.fit(X_train, y_train)
print("Model trained!")

location_name = "Maha Kali Amman Temple, Sri Lanka"
date = "2026-01-30"

# Get coordinates
geolocator = Nominatim(user_agent="weather_test")
location = geolocator.geocode(location_name)
lat, lng = location.latitude, location.longitude

# Fetch weather data
url = "https://api.open-meteo.com/v1/forecast"
params = {
    'latitude': lat,
    'longitude': lng,
    'daily': ['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
              'apparent_temperature_mean', 'precipitation_sum', 'rain_sum',
              'precipitation_hours', 'windspeed_10m_max', 'weathercode'],
    'start_date': date,
    'end_date': date
}

response = requests.get(url, params=params)
data = response.json()
daily = data['daily']

# Prepare features
import datetime as dt

date_obj = dt.datetime.strptime(date, '%Y-%m-%d')

test_data = {
    'location_lat': lat,
    'location_lng': lng,
    'elevation_x': data.get('elevation', 0),
    'station_lat': lat,
    'station_lng': lng,
    'distance_km': 0,
    'month': date_obj.month,
    'day': date_obj.day,
    'dayofweek': date_obj.weekday(),
    'temperature_2m_mean': daily['temperature_2m_mean'][0],
    'temperature_2m_max': daily['temperature_2m_max'][0],
    'temperature_2m_min': daily['temperature_2m_min'][0],
    'apparent_temperature_mean': daily['apparent_temperature_mean'][0],
    'precipitation_sum': daily['precipitation_sum'][0],
    'rain_sum': daily['rain_sum'][0],
    'precipitation_hours': daily['precipitation_hours'][0],
    'windspeed_10m_max': daily['windspeed_10m_max'][0]
}

X_new = pd.DataFrame([test_data])

# Predict
prediction = model.predict(X_new)[0]
weather_condition = le.inverse_transform([prediction])[0]
proba = model.predict_proba(X_new)[0]
confidence = proba[prediction] * 100

# Calculate travel score
base_scores = {'Clear': 95, 'Cloudy': 75, 'Fog': 50, 'Rain': 30, 'Heavy Rain': 10, 'Other': 40}
score = base_scores[weather_condition]

temp = test_data['temperature_2m_mean']
if 15 <= temp <= 28:
    score += 5
elif 10 <= temp < 15 or 28 < temp <= 32:
    score -= 5
elif temp < 10 or temp > 32:
    score -= 15

if test_data['precipitation_sum'] > 10:
    score -= 20
elif test_data['precipitation_sum'] > 5:
    score -= 10
elif test_data['precipitation_sum'] > 0:
    score -= 5

score = max(0, min(100, score))

# Results
print(f"\n{'=' * 60}")
print(f"Location: {location_name}")
print(f"Date: {date}")
print(f"{'=' * 60}")
print(f"Weather: {weather_condition} (Confidence: {confidence:.1f}%)")
print(f"TRAVEL SCORE: {score}/100")
print(f"{'=' * 60}")
print(f"Temperature: {temp:.1f}°C")
print(f"Precipitation: {test_data['precipitation_sum']:.1f}mm")
print(f"Wind: {test_data['windspeed_10m_max']:.1f}km/h")

if score >= 80:
    print(f"\n Excellent conditions for travel!")
elif score >= 65:
    print(f"\n Good conditions for travel.")
elif score >= 50:
    print(f"\n️ Fair conditions - plan accordingly.")
else:
    print(f"\n Not ideal for travel.")
print(f"{'=' * 60}\n")

Model trained!


AttributeError: 'NoneType' object has no attribute 'latitude'